# Markov categories as executable string diagrams

This notebook connects the finite numerical objects in `markov_entropy` with the free Markov-category diagrams in DisCoPy. DisCoPy is used for symbolic structure and drawing; NumPy matrices remain the numerical source of truth.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from markov_entropy import Channel, Distribution, FiniteSpace, entropy, mutual_information
from markov_entropy.adapters import DiscopyAdapter
from markov_entropy.divergences import KL

## Finite spaces and stochastic channels

Each atomic finite space becomes one wire. A product space becomes a tensor product of wires.

In [ ]:
X = FiniteSpace([0, 1])
Y = FiniteSpace(['a', 'b'])
p = Distribution(X, [0.4, 0.6])
f = Channel(X, Y, [[0.8, 0.1], [0.2, 0.9]])
adapter = DiscopyAdapter({X: 'X', Y: 'Y'})
assert str(adapter.type_for(X.tensor(Y))) == 'X @ Y'

## Copy, discard, composition, and tensor

The adapter uses DisCoPy's canonical copy and discard operations rather than ordinary generic boxes.

In [ ]:
channel_diagram = adapter.channel_box(f, 'f')
copy_diagram = adapter.copy_diagram(X)
discard_diagram = adapter.discard_diagram(X)
tensor_diagram = adapter.tensor_diagram(f, f, 'f', 'f')
assert channel_diagram.dom == adapter.type_for(X)
assert copy_diagram.cod == adapter.type_for(X.tensor(X))
assert len(discard_diagram.cod) == 0
assert tensor_diagram.dom == adapter.type_for(X.tensor(X))

## Entropy: copied output versus independent copies

For KL divergence, the numerical distance between these two processes recovers Shannon entropy.

In [ ]:
entropy_pair = adapter.entropy_diagrams(p)
assert entropy_pair.actual.dom == entropy_pair.reference.dom
assert entropy_pair.actual.cod == entropy_pair.reference.cod
assert entropy(p, KL()) > 0

## Mutual information: joint state versus product marginals

In [ ]:
joint = Distribution(X.tensor(Y), [0.35, 0.05, 0.10, 0.50])
mi_pair = adapter.mutual_information_diagrams(joint)
assert mi_pair.actual.dom == mi_pair.reference.dom
assert mi_pair.actual.cod == mi_pair.reference.cod
assert mutual_information(joint, KL()) > 0

## Rendering

`DiagramComparison.draw` writes both sides separately so they can be placed side-by-side in papers and documentation.

In [ ]:
with TemporaryDirectory() as directory:
    actual = Path(directory) / 'entropy-actual.svg'
    reference = Path(directory) / 'entropy-reference.svg'
    entropy_pair.draw(str(actual), str(reference))
    assert actual.exists() and actual.stat().st_size > 0
    assert reference.exists() and reference.stat().st_size > 0

Run `uv run python examples/render_discopy_gallery.py` to regenerate the complete documentation gallery.